# 04 — Dashboard Data Preparation
## AuctionIQ — IPL Player Valuation & Auction Intelligence
**Goal:** Prepare all export CSVs for Power BI — season trends, team analysis, player cards, and auction valuation data.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('../exports', exist_ok=True)

# Load all processed data
master  = pd.read_csv('../data/processed/master_balls.csv')
meta    = pd.read_csv('../data/processed/master_meta.csv')
players = pd.read_csv('../data/processed/player_index.csv')
batting = pd.read_csv('../data/processed/batting_stats.csv')
bowling = pd.read_csv('../data/processed/bowling_stats.csv')

master['season'] = master['season'].astype(str)
meta['season']   = meta['season'].astype(str)

print(f"✅ All data loaded")
print(f"   Master balls : {len(master):,}")
print(f"   Matches      : {len(meta):,}")
print(f"   Players      : {len(players)}")
print(f"   Batters      : {len(batting)}")
print(f"   Bowlers      : {len(bowling)}")

✅ All data loaded
   Master balls : 287,513
   Matches      : 1,209
   Players      : 306
   Batters      : 229
   Bowlers      : 191


In [2]:
# Season trends — for Power BI line charts
legal = master[master['wides'].isna()].copy()

season_trends = legal.groupby('season').agg(
    total_runs      = ('runs_off_bat', 'sum'),
    total_balls     = ('ball', 'count'),
    total_matches   = ('match_id', 'nunique'),
    total_fours     = ('runs_off_bat', lambda x: (x==4).sum()),
    total_sixes     = ('runs_off_bat', lambda x: (x==6).sum()),
    dot_balls       = ('runs_off_bat', lambda x: (x==0).sum()),
).reset_index()

# Wickets per season
wickets_season = master[
    master['wicket_type'].notna() &
    ~master['wicket_type'].isin(['run out','retired hurt'])
].groupby('season').size().reset_index(name='total_wickets')

season_trends = season_trends.merge(wickets_season, on='season', how='left')

# Derived
season_trends['avg_run_rate']    = (season_trends['total_runs'] /
                                   (season_trends['total_balls']/6)).round(2)
season_trends['avg_match_score'] = (season_trends['total_runs'] /
                                    season_trends['total_matches']).round(0)
season_trends['six_rate']        = (season_trends['total_sixes'] /
                                    season_trends['total_balls'] * 100).round(2)
season_trends['dot_ball_pct']    = (season_trends['dot_balls'] /
                                    season_trends['total_balls'] * 100).round(2)

# Sort seasons properly
season_order = ['2007/08','2009','2009/10','2011','2012','2013',
                '2014','2015','2016','2017','2018','2019',
                '2020/21','2021','2022','2023','2024','2025','2026']
season_trends['season_order'] = season_trends['season'].map(
    {s:i for i,s in enumerate(season_order)}
)
season_trends = season_trends.sort_values('season_order').drop('season_order', axis=1)

print(f"✅ Season trends built: {len(season_trends)} seasons")
print(season_trends[['season','total_matches','avg_run_rate',
                      'avg_match_score','six_rate']].to_string(index=False))

✅ Season trends built: 19 seasons
 season  total_matches  avg_run_rate  avg_match_score  six_rate
2007/08             58          7.74            290.0      4.78
   2009             57          7.01            270.0      3.86
2009/10             60          7.61            296.0      4.19
   2011             73          7.24            273.0      3.87
   2012             74          7.41            288.0      4.24
   2013             76          7.28            283.0      3.85
   2014             60          7.75            299.0      5.14
   2015             59          7.92            295.0      5.24
   2016             60          7.89            299.0      4.68
   2017             59          8.00            304.0      5.25
   2018             60          8.28            318.0      6.30
   2019             60          8.02            310.0      5.65
2020/21             60          7.90            309.0      5.22
   2021             60          7.62            295.0      4.92
   202

In [3]:
# Team wins and performance
team_wins = meta.groupby('winner').agg(
    wins = ('match_id', 'count')
).reset_index().rename(columns={'winner':'team'})

team_matches = pd.melt(
    meta[['match_id','toss_winner','winner']],
    id_vars=['match_id','winner'],
    value_vars=['toss_winner'],
    value_name='team'
)[['match_id','team','winner']].copy()

# Count total matches per team from master
team_batting = master.groupby('batting_team')['match_id'].nunique().reset_index()
team_batting.columns = ['team','total_matches']

team_stats = team_batting.merge(team_wins, on='team', how='left')
team_stats['wins'] = team_stats['wins'].fillna(0).astype(int)
team_stats['win_rate'] = (team_stats['wins'] / team_stats['total_matches'] * 100).round(2)

# Team batting stats
team_batting_stats = legal.groupby('batting_team').agg(
    total_runs   = ('runs_off_bat', 'sum'),
    total_balls  = ('ball', 'count'),
    total_sixes  = ('runs_off_bat', lambda x: (x==6).sum()),
    total_fours  = ('runs_off_bat', lambda x: (x==4).sum()),
).reset_index()

team_batting_stats['run_rate'] = (team_batting_stats['total_runs'] /
                                  (team_batting_stats['total_balls']/6)).round(2)
team_batting_stats.rename(columns={'batting_team':'team'}, inplace=True)
team_stats = team_stats.merge(team_batting_stats, on='team', how='left')

# Filter active/major teams only
major_teams = [
    'Mumbai Indians', 'Chennai Super Kings', 'Kolkata Knight Riders',
    'Royal Challengers Bangalore', 'Sunrisers Hyderabad',
    'Rajasthan Royals', 'Delhi Capitals', 'Punjab Kings',
    'Lucknow Super Giants', 'Gujarat Titans',
    'Royal Challengers Bengaluru'
]
team_stats = team_stats[team_stats['team'].isin(major_teams)]
team_stats = team_stats.sort_values('wins', ascending=False)

print(f"✅ Team stats built")
print(team_stats[['team','total_matches','wins','win_rate','run_rate']].to_string(index=False))

✅ Team stats built
                       team  total_matches  wins  win_rate  run_rate
             Mumbai Indians            284   153     53.87      7.99
        Chennai Super Kings            259   145     55.98      7.99
      Kolkata Knight Riders            272   136     50.00      7.87
           Rajasthan Royals            243   120     49.38      7.95
Royal Challengers Bangalore            240   114     47.50      7.86
        Sunrisers Hyderabad            203    98     48.28      8.03
             Delhi Capitals            113    54     47.79      8.21
             Gujarat Titans             68    41     60.29      8.64
               Punjab Kings             81    40     49.38      8.78
       Lucknow Super Giants             66    32     48.48      8.39
Royal Challengers Bengaluru             38    24     63.16      9.64


In [4]:
# Player of Match analysis — who wins games
potm = meta['player_of_match'].value_counts().reset_index()
potm.columns = ['player', 'potm_awards']

# Merge with player index
potm_merged = potm.merge(
    players[['player','role','PVS','BPI','BoPI']],
    on='player', how='left'
)

print(f"✅ Player of Match analysis built")
print(f"\nTop 20 match winners:")
print(potm_merged.head(20)[
    ['player','potm_awards','role','PVS']
].to_string(index=False))

✅ Player of Match analysis built

Top 20 match winners:
        player  potm_awards       role   PVS
AB de Villiers           25     Batter 78.51
      CH Gayle           22     Batter 75.94
     RG Sharma           21     Batter 65.07
       V Kohli           20     Batter 70.50
      MS Dhoni           18     Batter 72.23
     DA Warner           18     Batter 69.18
     SP Narine           17 Allrounder 76.70
     RA Jadeja           17 Allrounder 69.27
      KL Rahul           16     Batter 71.19
     SR Watson           16 Allrounder 68.19
    AD Russell           16 Allrounder 73.23
     YK Pathan           16 Allrounder 65.99
    KA Pollard           14 Allrounder 61.73
    JC Buttler           14     Batter 74.86
      SK Raina           14 Allrounder 63.50
   Rashid Khan           13 Allrounder 73.38
     SV Samson           13     Batter 69.24
     G Gambhir           13     Batter 57.14
  Shubman Gill           13     Batter 70.12
     AM Rahane           13     Batter 62.04

In [5]:
# Export 1 — Season trends
season_trends.to_csv('../exports/season_trends.csv', index=False)

# Export 2 — Team stats
team_stats.to_csv('../exports/team_stats.csv', index=False)

# Export 3 — Player index (full)
players.to_csv('../exports/player_index.csv', index=False)

# Export 4 — Batting stats (top 100)
batting.nlargest(100, 'BPI').to_csv('../exports/batting_top100.csv', index=False)

# Export 5 — Bowling stats (top 100)
bowling.nlargest(100, 'BoPI').to_csv('../exports/bowling_top100.csv', index=False)

# Export 6 — Player of Match
potm_merged.to_csv('../exports/potm_awards.csv', index=False)

# Export 7 — Valuation data
from_players = players[players['valuation'].notna()][[
    'player','role','PVS','BPI','BoPI',
    'auction_price','value_ratio','valuation'
]]
from_players.to_csv('../exports/auction_valuation.csv', index=False)

# Export 8 — Phase batting for heatmap
phase_bat = pd.read_csv('../data/processed/batting_stats.csv')
phase_cols = ['striker','powerplay_strike_rate',
              'middle_strike_rate','death_strike_rate','BPI']
phase_bat[phase_cols].nlargest(30,'BPI').to_csv(
    '../exports/phase_batting.csv', index=False)

print("✅ All Power BI exports complete")
print(f"\n📁 exports/ folder:")
for f in sorted(os.listdir('../exports')):
    size = os.path.getsize(f'../exports/{f}')
    rows = len(pd.read_csv(f'../exports/{f}'))
    print(f"   {f:<35} {rows:>4} rows  {size/1024:.1f} KB")

✅ All Power BI exports complete

📁 exports/ folder:
   auction_valuation.csv                 30 rows  1.9 KB
   batting_top100.csv                   100 rows  12.5 KB
   bowling_top100.csv                   100 rows  12.5 KB
   phase_batting.csv                     30 rows  1.2 KB
   player_index.csv                     306 rows  26.0 KB
   potm_awards.csv                      318 rows  10.1 KB
   season_trends.csv                     19 rows  1.3 KB
   team_stats.csv                        11 rows  0.7 KB


In [6]:
print("=" * 60)
print("   AUCTIONIQ — DATA PIPELINE COMPLETE")
print("=" * 60)
print(f"""
📊 DATASET
   Seasons    : 19 (2007/08 → 2026)
   Matches    : 1,209
   Balls      : 287,513
   Batters    : 229 (min 20 innings)
   Bowlers    : 191 (min 20 matches)
   Players    : 306 ranked

🏏 KEY FINDINGS
   Top Batter (BPI)    : H Klaasen (86.12)
   Top Bowler (BoPI)   : JJ Bumrah (83.65)
   Top Allrounder      : SP Narine (76.70)
   Most Undervalued    : CV Varun (₹1.7Cr, ratio 47.28)
   Most Overvalued     : RG Sharma (ratio 4.07)
   Most Match Wins     : AB de Villiers (25 POTM)
   Highest Win Rate    : RCB Bengaluru (63.16%)
   Best Run Rate 2026  : 9.25 (highest ever)

📁 EXPORTS (8 files → Power BI ready)
   season_trends.csv      → IPL evolution charts
   team_stats.csv         → Team performance page
   player_index.csv       → Full player rankings
   batting_top100.csv     → Batting leaderboard
   bowling_top100.csv     → Bowling leaderboard
   auction_valuation.csv  → Valuation matrix
   potm_awards.csv        → Match winner analysis
   phase_batting.csv      → Phase heatmap

Next → Power BI Dashboard (3 pages)
""")
print("=" * 60)

   AUCTIONIQ — DATA PIPELINE COMPLETE

📊 DATASET
   Seasons    : 19 (2007/08 → 2026)
   Matches    : 1,209
   Balls      : 287,513
   Batters    : 229 (min 20 innings)
   Bowlers    : 191 (min 20 matches)
   Players    : 306 ranked

🏏 KEY FINDINGS
   Top Batter (BPI)    : H Klaasen (86.12)
   Top Bowler (BoPI)   : JJ Bumrah (83.65)
   Top Allrounder      : SP Narine (76.70)
   Most Undervalued    : CV Varun (₹1.7Cr, ratio 47.28)
   Most Overvalued     : RG Sharma (ratio 4.07)
   Most Match Wins     : AB de Villiers (25 POTM)
   Highest Win Rate    : RCB Bengaluru (63.16%)
   Best Run Rate 2026  : 9.25 (highest ever)

📁 EXPORTS (8 files → Power BI ready)
   season_trends.csv      → IPL evolution charts
   team_stats.csv         → Team performance page
   player_index.csv       → Full player rankings
   batting_top100.csv     → Batting leaderboard
   bowling_top100.csv     → Bowling leaderboard
   auction_valuation.csv  → Valuation matrix
   potm_awards.csv        → Match winner analysis

In [7]:
# Add season_order to season_trends export for Power BI sorting
season_order_map = {
    '2007/08':1, '2009':2, '2009/10':3,
    '2011':4, '2012':5, '2013':6,
    '2014':7, '2015':8, '2016':9,
    '2017':10, '2018':11, '2019':12,
    '2020/21':13, '2021':14, '2022':15,
    '2023':16, '2024':17, '2025':18, '2026':19
}

season_trends['season_order'] = season_trends['season'].map(season_order_map)
season_trends.to_csv('../exports/season_trends.csv', index=False)
print("✅ season_trends.csv updated with season_order column")
print(season_trends[['season','season_order']].to_string(index=False))

✅ season_trends.csv updated with season_order column
 season  season_order
2007/08             1
   2009             2
2009/10             3
   2011             4
   2012             5
   2013             6
   2014             7
   2015             8
   2016             9
   2017            10
   2018            11
   2019            12
2020/21            13
   2021            14
   2022            15
   2023            16
   2024            17
   2025            18
   2026            19
